# Ablación: importancia por componente


In [4]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PAPER_STYLE = {
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "normal",
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "lines.linewidth": 1.6,
    "patch.linewidth": 0.45,
}

DATASET_LABELS = {
    "explicit_mars": "MARS-Explicit",
    "implicit_mars": "MARS-Implicit",
    "itm": "ITM-Rec",
    "doris": "DORIS",
    "moocubex": "MOOCubeX",
    "mooccubex": "MOOCubeX",
    "coco": "COCO",
}

VAR2COMP = {
    "no_sequence": "Secuencia",
    "no_graph": "Grafo",
    "no_context": "Contexto",
    "no_features": "Atributos",
    "no_user_features": "Atributos usuario",
    "no_item_features": "Atributos ítem",
    "sum_fusion": "Fusión (suma)",
    "no_gcl": "Contraste",
    "no_item_bias": "Sesgo ítem",
    "dot_product": "Scoring",
}
COMP_ORDER = [
    "Secuencia",
    "Grafo",
    "Contexto",
    "Atributos",
    "Atributos usuario",
    "Atributos ítem",
    "Sesgo ítem",
    "Fusión (suma)",
    "Contraste",
    "Scoring",
]
EXCLUDE_VARIANTS = {"full", "base"}


In [5]:
def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "results").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("No encuentro la raiz del repositorio EDuRec.")


def safe_name(text):
    return re.sub(r"[^a-zA-Z0-9]+", "_", str(text).lower()).strip("_")


def dataset_label(name):
    return DATASET_LABELS.get(str(name).lower(), str(name).replace("_", " ").title())


def metric_tag(metric):
    return str(metric).replace("@", "")


def component_name(variant):
    return VAR2COMP.get(str(variant), str(variant).replace("_", " ").title())


def load_importance(results_dir, metric):
    """Importancia de cada variante como caída media ± std frente al modelo completo."""
    frames = []
    for path in sorted(Path(results_dir).glob("*/ablation_results.csv")):
        df = pd.read_csv(path)
        if df.empty or metric not in df.columns:
            continue
        df = df[["variant", "seed", metric]].copy()
        df["dataset"] = dataset_label(path.parent.name)
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No hay ablation_results.csv en {results_dir}")

    seeds = pd.concat(frames, ignore_index=True)
    full_by_dataset = {
        dataset: group.set_index("seed")[metric]
        for dataset, group in seeds[seeds["variant"] == "full"].groupby("dataset")
    }

    rows = []
    for (dataset, variant), group in seeds.groupby(["dataset", "variant"]):
        if variant in EXCLUDE_VARIANTS:
            continue
        full = full_by_dataset.get(dataset)
        if full is None:
            continue
        values = group.set_index("seed")[metric]
        common = full.index.intersection(values.index)
        if len(common) == 0:
            continue
        drop = full.loc[common] - values.loc[common]
        rows.append(
            {
                "dataset": dataset,
                "variant": variant,
                "component": component_name(variant),
                "importance_mean": float(drop.mean()),
                "importance_std": float(drop.std(ddof=1)) if len(drop) > 1 else 0.0,
                "n_seeds": int(len(drop)),
            }
        )

    importance = pd.DataFrame(rows)
    importance["component"] = pd.Categorical(
        importance["component"], categories=COMP_ORDER, ordered=True
    )
    return importance.sort_values(["dataset", "component", "variant"]).reset_index(drop=True)


def export_table(importance, outdir, metric):
    outdir.mkdir(parents=True, exist_ok=True)
    table = importance.copy()
    table["importance"] = [
        f"{row.importance_mean:.4f} ± {row.importance_std:.4f}"
        for row in table.itertuples()
    ]
    table.to_csv(outdir / f"importance_values_{metric_tag(metric)}.csv", index=False)


In [6]:
def savefig(outdir, name, formats, dpi=300):
    outdir.mkdir(parents=True, exist_ok=True)
    for fmt in formats:
        plt.savefig(outdir / f"{name}.{fmt}", bbox_inches="tight", dpi=dpi)


def apply_plot_style():
    sns.set_theme(
        context="paper",
        style="whitegrid",
        palette="bright",
        font="DejaVu Sans",
        rc=PAPER_STYLE,
    )


def style_axis(ax, grid_axis):
    ax.set_axisbelow(True)
    ax.grid(axis=grid_axis, color="0.88", linewidth=0.7)
    ax.grid(axis="x" if grid_axis == "y" else "y", visible=False)
    sns.despine(ax=ax, trim=True)
    ax.tick_params(length=3, width=0.7, pad=5)


def overlay_std_errorbars(ax, data, x, hue, order, palette, value_col, std_col):
    """Dibuja barras de error (desviación típica) sobre líneas ya trazadas."""
    lookup = {
        (getattr(row, x), getattr(row, hue)): (
            getattr(row, value_col),
            getattr(row, std_col),
        )
        for row in data.itertuples()
    }
    categories = [tick.get_text() for tick in ax.get_xticklabels()]
    lines = [line for line in ax.lines if len(np.atleast_1d(line.get_xdata()))]
    for line, level in zip(lines, order):
        for xpos in line.get_xdata():
            idx = int(round(float(xpos)))
            if not 0 <= idx < len(categories):
                continue
            value, error = lookup.get((categories[idx], level), (np.nan, np.nan))
            if pd.isna(value) or pd.isna(error):
                continue
            ax.errorbar(
                xpos,
                value,
                yerr=error,
                color=palette.get(level, "0.2"),
                capsize=1.8,
                elinewidth=0.65,
                capthick=0.65,
                fmt="none",
                zorder=4,
            )


def impact_range(mean, error):
    mean = np.asarray(mean, dtype=float)
    error = np.asarray(error, dtype=float)
    return np.concatenate([mean + error, mean - error])


def set_lims(ax, values, axis):
    finite = np.asarray([v for v in values if np.isfinite(v)], dtype=float)
    if finite.size == 0:
        return
    spread = max(float(finite.max()) - float(finite.min()), 0.02)
    pad = spread * 0.16
    getattr(ax, f"set_{axis}lim")(
        min(0.0, float(finite.min()) - pad), max(0.0, float(finite.max()) + pad)
    )


def pivot_importance(importance, values):
    return (
        importance.pivot(index="component", columns="dataset", values=values)
        .reindex(COMP_ORDER)
        .dropna(how="all")
    )


def plot_grouped_bar(importance, outdir, metric, formats, show_error_bars=False):
    mean = pivot_importance(importance, "importance_mean")
    std = pivot_importance(importance, "importance_std").reindex(mean.index)
    datasets = list(mean.columns)
    apply_plot_style()
    fig, ax = plt.subplots(
        figsize=(max(7.2, 0.85 * len(mean.index) + 2.2), 3.8), layout="constrained"
    )
    sns.barplot(
        data=importance[importance["component"].isin(mean.index)],
        x="component",
        y="importance_mean",
        hue="dataset",
        order=list(mean.index),
        hue_order=datasets,
        errorbar=None,
        edgecolor="0.25",
        linewidth=0.45,
        ax=ax,
    )
    if show_error_bars:
        components = list(mean.index)
        lookup = {
            (row.component, row.dataset): (row.importance_mean, row.importance_std)
            for row in importance.itertuples()
        }
        for container, dataset in zip(ax.containers, datasets):
            for bar in container:
                index = int(round(bar.get_x() + bar.get_width() / 2))
                if not 0 <= index < len(components):
                    continue
                value, error = lookup.get(
                    (components[index], dataset), (np.nan, np.nan)
                )
                if pd.isna(value) or pd.isna(error):
                    continue
                ax.errorbar(
                    bar.get_x() + bar.get_width() / 2,
                    value,
                    yerr=error,
                    color="0.15",
                    capsize=2.5,
                    elinewidth=0.8,
                    capthick=0.8,
                    fmt="none",
                )
    ax.axhline(0, color="0.25", linewidth=0.8)
    ax.set_ylabel(f"Caída {metric.upper()} frente al modelo completo")
    ax.set_title(f"Importancia por componente ({metric.upper()})")
    span = importance["importance_mean"].values
    if show_error_bars:
        span = impact_range(span, importance["importance_std"].values)
    set_lims(ax, span, "y")
    style_axis(ax, "y")
    ax.legend(
        title=None, frameon=False, ncol=min(len(datasets), 3), loc="upper right"
    )
    savefig(outdir, f"importance_grouped_bar_{metric_tag(metric)}", formats)
    plt.close()


def plot_line(importance, outdir, metric, formats):
    mean = pivot_importance(importance, "importance_mean")
    datasets = list(mean.columns)
    palette = dict(zip(datasets, sns.color_palette("bright", len(datasets))))
    subset = importance[importance["component"].isin(mean.index)]
    apply_plot_style()
    fig, ax = plt.subplots(
        figsize=(max(7.2, 0.85 * len(mean.index) + 2.2), 3.8), layout="constrained"
    )
    sns.lineplot(
        data=subset,
        x="component",
        y="importance_mean",
        hue="dataset",
        style="dataset",
        markers=True,
        dashes=False,
        hue_order=datasets,
        sort=False,
        errorbar=None,
        palette=palette,
        ax=ax,
    )
    overlay_std_errorbars(
        ax,
        subset,
        "component",
        "dataset",
        datasets,
        palette,
        "importance_mean",
        "importance_std",
    )
    ax.axhline(0, color="0.25", linewidth=0.8)
    ax.set_ylabel(f"Caída {metric.upper()} frente al modelo completo")
    ax.set_title(f"Perfil de ablación ({metric.upper()})")
    set_lims(
        ax,
        impact_range(
            importance["importance_mean"].values,
            importance["importance_std"].values,
        ),
        "y",
    )
    style_axis(ax, "y")
    ax.legend(
        title=None,
        frameon=False,
        ncol=min(len(mean.columns), 3),
        loc="upper right",
    )
    savefig(outdir, f"importance_line_{metric_tag(metric)}", formats)
    plt.close()


def plot_individual_barh(importance, outdir, metric, formats, show_error_bars=False, sort_by="impact"):
    for dataset, subset in importance.groupby("dataset", sort=True):
        subset = subset.sort_values(
            "importance_mean" if sort_by == "impact" else "component",
            ascending=sort_by == "impact",
        )
        labels = subset["component"].astype(str).tolist()
        indexed = subset.set_index("component")
        apply_plot_style()
        fig, ax = plt.subplots(
            figsize=(5.8, max(3.2, 0.42 * len(subset) + 1.1)), layout="constrained"
        )
        sns.barplot(
            data=subset,
            x="importance_mean",
            y="component",
            order=labels,
            errorbar=None,
            color=sns.color_palette("colorblind")[0],
            edgecolor="0.25",
            linewidth=0.45,
            ax=ax,
        )
        if show_error_bars:
            for bar, component in zip(ax.containers[0], labels):
                error = indexed.loc[component, "importance_std"]
                if pd.notna(error):
                    ax.errorbar(
                        indexed.loc[component, "importance_mean"],
                        bar.get_y() + bar.get_height() / 2,
                        xerr=error,
                        color="0.15",
                        capsize=2.5,
                        elinewidth=0.8,
                        capthick=0.8,
                        fmt="none",
                    )
        ax.axvline(0, color="0.25", linewidth=0.8)
        ax.set_xlabel(f"Caída {metric.upper()} frente al modelo completo")
        ax.set_title(f"{dataset} - importancia por componente")
        span = subset["importance_mean"].values
        if show_error_bars:
            span = impact_range(span, subset["importance_std"].values)
        set_lims(ax, span, "x")
        style_axis(ax, "x")
        safe = safe_name(dataset)
        savefig(
            outdir / safe,
            f"importance_barh_{safe}_{metric_tag(metric)}",
            formats,
        )
        plt.close()


## Configuración


In [7]:
PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results" / "ablations"
PLOTS_DIR = PROJECT_ROOT / "results" / "plots" / "ablation"
TABLES_DIR = PROJECT_ROOT / "results" / "tables" / "ablation"
METRIC = "ndcg@20"
FORMATS = ["png", "pdf"]
SHOW_ERROR_BARS = True
SORT_INDIVIDUAL = "impact"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Resultados: {RESULTS_DIR}")
print(f"Figuras:    {PLOTS_DIR}")
print(f"Tablas:     {TABLES_DIR}")


Resultados: /home/pacatro/Code/projects/EDuRec/results/ablations
Figuras:    /home/pacatro/Code/projects/EDuRec/results/plots/ablation
Tablas:     /home/pacatro/Code/projects/EDuRec/results/tables/ablation


## Cargar resultados y calcular media ± std


In [8]:
importance = load_importance(RESULTS_DIR, METRIC)
if importance.empty:
    raise ValueError("No hay variantes de ablacion que analizar.")
importance[
    ["dataset", "component", "variant", "importance_mean", "importance_std", "n_seeds"]
]


,dataset,component,variant,importance_mean,importance_std,n_seeds
0,DORIS,Secuencia,no_sequence,3.942649e-03,5.453844e-03,3
1,DORIS,Grafo,no_graph,-2.703230e-03,1.935408e-03,3
2,DORIS,Contexto,no_context,6.804522e-01,6.524325e-03,3
3,DORIS,Atributos,no_features,7.935941e-03,7.523728e-03,3
4,DORIS,Atributos usuario,no_user_features,-2.126137e-03,9.697899e-03,3
5,DORIS,Atributos ítem,no_item_features,7.650733e-03,7.002225e-03,3
6,DORIS,Sesgo ítem,no_item_bias,-1.745363e-03,7.463398e-03,3
7,DORIS,Fusión (suma),sum_fusion,2.536317e-03,5.062841e-03,3
8,DORIS,Contraste,no_gcl,-2.114594e-03,3.839615e-03,3
9,DORIS,Scoring,dot_product,4.773617e-03,1.017480e-02,3


## Exportar tabla y figuras


In [9]:
export_table(importance, TABLES_DIR, METRIC)
plot_grouped_bar(importance, PLOTS_DIR, METRIC, FORMATS, show_error_bars=SHOW_ERROR_BARS)
plot_line(importance, PLOTS_DIR, METRIC, FORMATS)
plot_individual_barh(
    importance,
    PLOTS_DIR,
    METRIC,
    FORMATS,
    show_error_bars=SHOW_ERROR_BARS,
    sort_by=SORT_INDIVIDUAL,
)

files = sorted(p.relative_to(PLOTS_DIR) for p in PLOTS_DIR.rglob("*") if p.is_file())
print(f"Generadas {len(files)} figuras en {PLOTS_DIR}")
files


Generadas 14 figuras en /home/pacatro/Code/projects/EDuRec/results/plots/ablation


[PosixPath('doris/importance_barh_doris_ndcg20.pdf'),
 PosixPath('doris/importance_barh_doris_ndcg20.png'),
 PosixPath('importance_grouped_bar_ndcg20.pdf'),
 PosixPath('importance_grouped_bar_ndcg20.png'),
 PosixPath('importance_line_ndcg20.pdf'),
 PosixPath('importance_line_ndcg20.png'),
 PosixPath('itm_rec/importance_barh_itm_rec_ndcg20.pdf'),
 PosixPath('itm_rec/importance_barh_itm_rec_ndcg20.png'),
 PosixPath('mars_explicit/importance_barh_mars_explicit_ndcg20.pdf'),
 PosixPath('mars_explicit/importance_barh_mars_explicit_ndcg20.png'),
 PosixPath('mars_implicit/importance_barh_mars_implicit_ndcg20.pdf'),
 PosixPath('mars_implicit/importance_barh_mars_implicit_ndcg20.png'),
 PosixPath('moocubex/importance_barh_moocubex_ndcg20.pdf'),
 PosixPath('moocubex/importance_barh_moocubex_ndcg20.png')]